# Session 8.2: Sentiment Analysis - Movie Reviews

## Learning Objectives
1. Understand Bag of Words (BoW) representation
2. Master TF-IDF vectorization
3. Build sentiment analyzer with traditional ML
4. Compare Naive Bayes and Logistic Regression
5. Achieve >85% accuracy on IMDB reviews

**Duration:** 2 hours  
**Target:** Build sentiment analyzer that recalls Module 0 demo!

## 1. Setup and Data Loading

### LLM Prompt for Installation:
```
Create a Python code block that installs required libraries for sentiment analysis:
scikit-learn, nltk, pandas, numpy, matplotlib, seaborn, and wordcloud.
```

In [ ]:
# Install required libraries
!pip install scikit-learn nltk pandas numpy matplotlib seaborn wordcloud -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from wordcloud import WordCloud

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK data
for data in ['punkt', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(data, quiet=True)

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

### Load IMDB Movie Reviews Dataset

We'll use the IMDB dataset with 50,000 movie reviews.

In [ ]:
# Load IMDB dataset
# Option 1: Load from TensorFlow datasets
try:
    import tensorflow as tf
    import tensorflow_datasets as tfds
    
    # Load dataset
    dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)
    train_dataset, test_dataset = dataset['train'], dataset['test']
    
    # Convert to lists
    train_texts, train_labels = [], []
    for text, label in tfds.as_numpy(train_dataset):
        train_texts.append(text.decode('utf-8'))
        train_labels.append(int(label))
    
    test_texts, test_labels = [], []
    for text, label in tfds.as_numpy(test_dataset):
        test_texts.append(text.decode('utf-8'))
        test_labels.append(int(label))
    
    print("✓ Dataset loaded from TensorFlow Datasets")
    
except:
    # Option 2: Create sample dataset
    print("Creating sample dataset...")
    
    # Sample positive and negative reviews
    positive_reviews = [
        "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.",
        "I loved every minute of this film. Best movie I've seen all year! Highly recommended.",
        "Brilliant performances and stunning cinematography. A masterpiece!",
        "Amazing story with great character development. 10/10 would watch again.",
        "Exceptional film with outstanding direction. Truly a work of art."
    ] * 500  # Repeat to create larger dataset
    
    negative_reviews = [
        "Terrible movie. Waste of time and money. Awful acting and predictable plot.",
        "I hated this film. Boring, slow, and completely unoriginal.",
        "Worst movie ever. I walked out halfway through. Don't waste your time.",
        "Horrible acting and terrible script. I want my money back.",
        "Disappointing and poorly executed. Not worth watching."
    ] * 500
    
    # Combine and create labels
    all_reviews = positive_reviews + negative_reviews
    all_labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)
    
    # Split into train and test
    from sklearn.model_selection import train_test_split
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        all_reviews, all_labels, test_size=0.2, random_state=42, stratify=all_labels
    )
    
    print("✓ Sample dataset created")

# Display dataset info
print(f"\nDataset Statistics:")
print(f"Training samples: {len(train_texts)}")
print(f"Test samples: {len(test_texts)}")
print(f"Positive reviews (train): {sum(train_labels)}")
print(f"Negative reviews (train): {len(train_labels) - sum(train_labels)}")

In [ ]:
# Display sample reviews
print("SAMPLE REVIEWS:\n")
print("Positive Review:")
print(train_texts[0][:300] + "...")
print(f"\nLabel: {train_labels[0]} (1 = Positive)")

print("\n" + "="*80 + "\n")

# Find a negative review
neg_idx = [i for i, label in enumerate(train_labels) if label == 0][0]
print("Negative Review:")
print(train_texts[neg_idx][:300] + "...")
print(f"\nLabel: {train_labels[neg_idx]} (0 = Negative)")

## 2. Text Preprocessing

Apply preprocessing pipeline from Session 8.1

In [ ]:
def preprocess_text(text):
    """
    Preprocess text for sentiment analysis.
    
    Args:
        text (str): Raw text
    
    Returns:
        str: Cleaned text
    """
    # Lowercase
    text = text.lower()
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Preprocess all reviews
print("Preprocessing reviews...")
train_texts_clean = [preprocess_text(text) for text in train_texts]
test_texts_clean = [preprocess_text(text) for text in test_texts]

print("✓ Preprocessing complete!")

# Show before/after
print("\nBEFORE:")
print(train_texts[0][:200])
print("\nAFTER:")
print(train_texts_clean[0][:200])

## 3. Exploratory Data Analysis

### 3.1 Class Distribution

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
train_counts = pd.Series(train_labels).value_counts().sort_index()
axes[0].bar(['Negative', 'Positive'], train_counts.values, color=['#d62728', '#2ca02c'])
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Training Set: Class Distribution', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(train_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')

# Test set
test_counts = pd.Series(test_labels).value_counts().sort_index()
axes[1].bar(['Negative', 'Positive'], test_counts.values, color=['#d62728', '#2ca02c'])
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Test Set: Class Distribution', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(test_counts.values):
    axes[1].text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Dataset is {'balanced' if abs(train_counts[0] - train_counts[1]) < 100 else 'imbalanced'}!")

### 3.2 Word Clouds - Positive vs Negative

In [ ]:
# Separate positive and negative reviews
positive_texts = [train_texts_clean[i] for i, label in enumerate(train_labels) if label == 1]
negative_texts = [train_texts_clean[i] for i, label in enumerate(train_labels) if label == 0]

# Create word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive reviews
positive_text = ' '.join(positive_texts[:1000])  # Use subset for speed
wordcloud_pos = WordCloud(width=800, height=400, 
                          background_color='white',
                          colormap='Greens',
                          stopwords=set(stopwords.words('english'))).generate(positive_text)

axes[0].imshow(wordcloud_pos, interpolation='bilinear')
axes[0].set_title('Word Cloud: Positive Reviews', fontsize=14, fontweight='bold', color='green')
axes[0].axis('off')

# Negative reviews
negative_text = ' '.join(negative_texts[:1000])
wordcloud_neg = WordCloud(width=800, height=400,
                          background_color='white',
                          colormap='Reds',
                          stopwords=set(stopwords.words('english'))).generate(negative_text)

axes[1].imshow(wordcloud_neg, interpolation='bilinear')
axes[1].set_title('Word Cloud: Negative Reviews', fontsize=14, fontweight='bold', color='red')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 3.3 Top Words - Positive vs Negative

In [ ]:
from collections import Counter

# Get top words
stop_words = set(stopwords.words('english'))

# Positive words
pos_words = ' '.join(positive_texts).split()
pos_words = [w for w in pos_words if w not in stop_words and len(w) > 2]
pos_freq = Counter(pos_words).most_common(15)

# Negative words
neg_words = ' '.join(negative_texts).split()
neg_words = [w for w in neg_words if w not in stop_words and len(w) > 2]
neg_freq = Counter(neg_words).most_common(15)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive
words, counts = zip(*pos_freq)
axes[0].barh(range(len(words)), counts, color='lightgreen')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].set_xlabel('Frequency', fontsize=11)
axes[0].set_title('Top 15 Words: Positive Reviews', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

# Negative
words, counts = zip(*neg_freq)
axes[1].barh(range(len(words)), counts, color='lightcoral')
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].set_xlabel('Frequency', fontsize=11)
axes[1].set_title('Top 15 Words: Negative Reviews', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Bag of Words (BoW) Representation

Convert text to numerical vectors using word counts.

In [ ]:
# Create Bag of Words vectorizer
bow_vectorizer = CountVectorizer(
    max_features=5000,  # Keep top 5000 words
    stop_words='english',  # Remove stop words
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2  # Word must appear in at least 2 documents
)

# Fit and transform training data
X_train_bow = bow_vectorizer.fit_transform(train_texts_clean)
X_test_bow = bow_vectorizer.transform(test_texts_clean)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

print("Bag of Words Features:")
print(f"Training shape: {X_train_bow.shape}")
print(f"Test shape: {X_test_bow.shape}")
print(f"Vocabulary size: {len(bow_vectorizer.vocabulary_)}")
print(f"Sparsity: {(1 - X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1])) * 100:.2f}%")

# Show sample features
feature_names = bow_vectorizer.get_feature_names_out()
print(f"\nSample features: {feature_names[:20]}")

### Visualize BoW for a Sample Review

In [ ]:
# Visualize BoW for first review
sample_idx = 0
sample_bow = X_train_bow[sample_idx].toarray()[0]

# Get non-zero features
nonzero_idx = np.nonzero(sample_bow)[0]
nonzero_features = [(feature_names[i], sample_bow[i]) for i in nonzero_idx]
nonzero_features = sorted(nonzero_features, key=lambda x: x[1], reverse=True)[:15]

print(f"Review: {train_texts_clean[sample_idx][:200]}...\n")
print("Top 15 BoW Features:")

words, counts = zip(*nonzero_features)
plt.figure(figsize=(12, 5))
plt.barh(range(len(words)), counts, color='skyblue')
plt.yticks(range(len(words)), words)
plt.xlabel('Count', fontsize=11)
plt.title('Bag of Words: Top Features for Sample Review', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Model 1: Naive Bayes with BoW

In [ ]:
# Train Naive Bayes classifier
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

# Make predictions
y_pred_nb = nb_model.predict(X_test_bow)

# Evaluate
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print("NAIVE BAYES + BOW RESULTS:\n")
print(f"Accuracy:  {accuracy_nb:.4f}")
print(f"Precision: {precision_nb:.4f}")
print(f"Recall:    {recall_nb:.4f}")
print(f"F1-Score:  {f1_nb:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))

### Confusion Matrix

In [ ]:
# Confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=11)
plt.ylabel('Actual', fontsize=11)
plt.title('Confusion Matrix: Naive Bayes + BoW', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"True Negatives: {cm_nb[0,0]}")
print(f"False Positives: {cm_nb[0,1]}")
print(f"False Negatives: {cm_nb[1,0]}")
print(f"True Positives: {cm_nb[1,1]}")

## 6. TF-IDF Representation

Term Frequency-Inverse Document Frequency gives more weight to important words.

In [ ]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True  # Use log scaling for TF
)

# Fit and transform
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts_clean)
X_test_tfidf = tfidf_vectorizer.transform(test_texts_clean)

print("TF-IDF Features:")
print(f"Training shape: {X_train_tfidf.shape}")
print(f"Test shape: {X_test_tfidf.shape}")

# Compare BoW vs TF-IDF for sample review
sample_tfidf = X_train_tfidf[sample_idx].toarray()[0]
nonzero_idx_tfidf = np.nonzero(sample_tfidf)[0]
nonzero_tfidf = [(feature_names[i], sample_tfidf[i]) for i in nonzero_idx_tfidf]
nonzero_tfidf = sorted(nonzero_tfidf, key=lambda x: x[1], reverse=True)[:15]

print(f"\nTop TF-IDF features for sample review:")
for word, score in nonzero_tfidf[:10]:
    print(f"{word:<20} {score:.4f}")

## 7. Model 2: Logistic Regression with TF-IDF

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0  # Regularization parameter
)

lr_model.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluate
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print("LOGISTIC REGRESSION + TF-IDF RESULTS:\n")
print(f"Accuracy:  {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall:    {recall_lr:.4f}")
print(f"F1-Score:  {f1_lr:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=11)
plt.ylabel('Actual', fontsize=11)
plt.title('Confusion Matrix: Logistic Regression + TF-IDF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Model Comparison

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': ['Naive Bayes + BoW', 'Logistic Regression + TF-IDF'],
    'Accuracy': [accuracy_nb, accuracy_lr],
    'Precision': [precision_nb, precision_lr],
    'Recall': [recall_nb, recall_lr],
    'F1-Score': [f1_nb, f1_lr]
})

print("MODEL COMPARISON:\n")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35

nb_scores = [accuracy_nb, precision_nb, recall_nb, f1_nb]
lr_scores = [accuracy_lr, precision_lr, recall_lr, f1_lr]

ax.bar(x - width/2, nb_scores, width, label='Naive Bayes + BoW', color='skyblue')
ax.bar(x + width/2, lr_scores, width, label='Logistic Regression + TF-IDF', color='lightcoral')

ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Comparison: Sentiment Analysis', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (nb, lr) in enumerate(zip(nb_scores, lr_scores)):
    ax.text(i - width/2, nb + 0.02, f'{nb:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + width/2, lr + 0.02, f'{lr:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nBest Model: {'Logistic Regression + TF-IDF' if accuracy_lr > accuracy_nb else 'Naive Bayes + BoW'}")
print(f"Best Accuracy: {max(accuracy_lr, accuracy_nb):.4f}")

## 9. Feature Importance

What words are most indicative of positive/negative sentiment?

In [ ]:
# Get feature importance from Logistic Regression
feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = lr_model.coef_[0]

# Get top positive and negative words
top_positive_idx = np.argsort(coefficients)[-20:]
top_negative_idx = np.argsort(coefficients)[:20]

top_positive = [(feature_names[i], coefficients[i]) for i in top_positive_idx]
top_negative = [(feature_names[i], coefficients[i]) for i in top_negative_idx]

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive words
words, coefs = zip(*top_positive)
axes[0].barh(range(len(words)), coefs, color='lightgreen')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].set_xlabel('Coefficient', fontsize=11)
axes[0].set_title('Top 20 Positive Sentiment Indicators', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

# Negative words
words, coefs = zip(*top_negative)
axes[1].barh(range(len(words)), coefs, color='lightcoral')
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].set_xlabel('Coefficient', fontsize=11)
axes[1].set_title('Top 20 Negative Sentiment Indicators', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 10. Test on Custom Reviews

Let's test our model on new reviews!

In [ ]:
# Custom movie reviews
custom_reviews = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. Complete waste of time. I want my money back.",
    "Pretty good movie with some great moments, but also some boring parts.",
    "Masterpiece! Best film I've seen in years. Highly recommend!",
    "Awful acting, terrible plot. One of the worst movies ever made.",
    "It was okay. Nothing special but not terrible either."
]

def predict_sentiment(review, model, vectorizer):
    """
    Predict sentiment for a review.
    
    Args:
        review (str): Movie review text
        model: Trained classifier
        vectorizer: Fitted vectorizer
    
    Returns:
        tuple: (prediction, probability)
    """
    # Preprocess
    cleaned = preprocess_text(review)
    
    # Vectorize
    vectorized = vectorizer.transform([cleaned])
    
    # Predict
    prediction = model.predict(vectorized)[0]
    probability = model.predict_proba(vectorized)[0]
    
    return prediction, probability

# Test on custom reviews
print("CUSTOM REVIEW PREDICTIONS:\n")
print("=" * 80)

for i, review in enumerate(custom_reviews, 1):
    pred, prob = predict_sentiment(review, lr_model, tfidf_vectorizer)
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = prob[pred] * 100
    
    print(f"\nReview {i}:")
    print(f"{review}")
    print(f"\nPrediction: {sentiment} (Confidence: {confidence:.1f}%)")
    print(f"Probabilities: Negative={prob[0]:.3f}, Positive={prob[1]:.3f}")
    print("=" * 80)

## 11. Compare to Module 0 Demo

Remember the sentiment analyzer from Module 0? **You just built it!**

In [ ]:
print("🎉 CONGRATULATIONS! 🎉\n")
print("You've successfully built a sentiment analyzer that:")
print(f"✓ Achieves {max(accuracy_lr, accuracy_nb)*100:.1f}% accuracy")
print("✓ Uses Bag of Words and TF-IDF representations")
print("✓ Compares multiple machine learning algorithms")
print("✓ Can analyze any movie review in real-time")
print("\nThis is the same technology behind:")
print("• Product review analysis on Amazon")
print("• Social media sentiment tracking")
print("• Customer feedback analysis")
print("• Brand monitoring")
print("\nYou now understand what was 'under the hood' in Module 0!")

## 12. Save Model for Later Use

In [ ]:
import joblib

# Save the best model and vectorizer
joblib.dump(lr_model, 'sentiment_model.pkl')
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

print("✓ Model saved as 'sentiment_model.pkl'")
print("✓ Vectorizer saved as 'tfidf_vectorizer.pkl'")
print("\nYou can load them later with:")
print("model = joblib.load('sentiment_model.pkl')")
print("vectorizer = joblib.load('tfidf_vectorizer.pkl')")

## Success Criteria

- ✅ Understand Bag of Words representation
- ✅ Master TF-IDF vectorization
- ✅ Build and compare Naive Bayes and Logistic Regression
- ✅ Achieve >85% accuracy on IMDB reviews
- ✅ Interpret feature importance
- ✅ Test on custom reviews
- ✅ Compare to Module 0 demo successfully

**Next Session:** Word Embeddings and Multi-Class Text Classification!